In [0]:
# No serverless, o Azure SDK não funciona (DNS sinkhole bloqueia rede externa).
# A solução é Unity Catalog Lakehouse Federation — sem pacotes extras.
# python-dotenv ainda é útil para ler credenciais de referência do .env
%pip install python-dotenv
dbutils.library.restartPython()

In [0]:
from dotenv import load_dotenv
import os

load_dotenv()

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT") or "internshipdatalake"
CONTAINER = os.getenv("CONTAINER") or "raw"

In [0]:
# ---------------------------------------------------------------------------
# No serverless, não é possível criar ClientSecretCredential do Azure SDK.
# O acesso aos dados é feito via Unity Catalog Lakehouse Federation,
# onde a conexão com o SQL Server é gerenciada pela infraestrutura do UC.
# ---------------------------------------------------------------------------

print("Conexão gerenciada pelo Unity Catalog (catálogo estrangeiro: sqlserver_catalog)")
print("Não é necessário ClientSecretCredential no serverless.")

In [0]:
# Em vez de DataLakeServiceClient (Azure SDK), listamos os schemas
# disponíveis no catálogo estrangeiro sqlserver_catalog
print("Schemas disponíveis no SQL Server (via Lakehouse Federation):")
spark.sql("SHOW SCHEMAS IN sqlserver_catalog").show()

In [0]:
# Em vez de listar arquivos do ADLS, listamos as tabelas do SQL Server
print("Tabelas disponíveis no schema dbo:")
spark.sql("SHOW TABLES IN sqlserver_catalog.dbo").show(30, truncate=False)

In [0]:
import sys

notebook_path = os.path.dirname(os.path.abspath(sys.argv[0]))
env_path = os.path.join(notebook_path, ".env")

load_dotenv(dotenv_path=env_path)

load_dotenv()

In [0]:
def ler_tabela_sqlserver(nome_schema, nome_tabela):
    """Lê uma tabela do SQL Server via Lakehouse Federation.

    No serverless, substitui a leitura de CSV do ADLS pela leitura
    direta das tabelas no SQL Server (catálogo estrangeiro sqlserver_catalog).
    """
    tabela = f"sqlserver_catalog.{nome_schema}.{nome_tabela}"
    print(f"Lendo: {tabela}")
    return spark.table(tabela)

In [0]:
# ---------------------------------------------------------------------------
# No serverless, as configurações fs.azure.account.* são BLOQUEADAS:
#   erro: CONFIG_NOT_AVAILABLE — Configuration fs.azure.account.auth.type is not available
#
# O Lakehouse Federation já gerencia a autenticação automaticamente.
# Basta ler as tabelas diretamente:
# ---------------------------------------------------------------------------

df_pedidos = ler_tabela_sqlserver("dbo", "ecommerce_pedidos")
df_clientes = ler_tabela_sqlserver("dbo", "ecommerce_clientes")

print(f"\nPedidos: {df_pedidos.count()} linhas")
print(f"Clientes: {df_clientes.count()} linhas")
display(df_pedidos.limit(5))

In [0]:
df_pedidos = ler_tabela_sqlserver("dbo", "ecommerce_pedidos")

In [0]:
display(df_pedidos)


In [0]:
df_pedidos.select(
    "valor_total",
    "valor_frete"
).summary()

In [0]:
total_registros = df_pedidos.count()
print(f"Total de registros: {total_registros}")

In [0]:
df_pedidos.groupBy("status_pedido").count().orderBy("count", ascending=False).show()

In [0]:
from pyspark.sql.functions import col, round as round_

df_pedidos.groupBy("status_pedido") \
    .count() \
    .withColumn("percentual", round_((col("count") / total_registros) * 100, 2)) \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
from pyspark.sql.functions import min as spark_min, max as spark_max

df_pedidos.select(
    spark_min("dt_pedido").alias("primeiro_pedido"),
    spark_max("dt_pedido").alias("ultimo_pedido")
).show()

In [0]:
from pyspark.sql.functions import col, datediff, min as spark_min, max as spark_max, avg, stddev, round as round_

df_pedidos = df_pedidos.withColumn(
    "dias_prazo_estimado",
    datediff(col("dt_ultima_atualizacao_status"), col("dt_pedido"))
)

df_pedidos.select(
    "dt_pedido",
    "dt_ultima_atualizacao_status",
    "dias_prazo_estimado"
).show()

print("Estatísticas sobre dias_prazo_estimado:")
df_pedidos.select(
    spark_min("dias_prazo_estimado").alias("minimo"),
    spark_max("dias_prazo_estimado").alias("maximo"),
    round_(avg("dias_prazo_estimado"), 2).alias("media"),
    round_(stddev("dias_prazo_estimado"), 2).alias("desvio_padrao")
).show()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

contagem_nulos = df_pedidos.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_pedidos.columns
])
contagem_nulos.show(truncate=False)

total_nulos = contagem_nulos.collect()[0]
total = sum(total_nulos[c] for c in df_pedidos.columns)
if total == 0:
    print("✅ Não há valores nulos no DataFrame.")
else:
    print(f"⚠️ Existem {total} valores nulos no total.")